# Predicting Denver Traffic Accident Severity

In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    classification_report,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

In [3]:
# Load data.
df = pd.read_csv("data/ODC_CRIME_TRAFFICACCIDENTS5YR_P_7255667086186507966.csv", low_memory=False)

# Create the target variable before dropping leakage columns.
# Create high risk variable.

# Fill NaN as 0 so the comparison is safe
si = df["SERIOUSLY_INJURED"].fillna(0)
fat = df["FATALITIES"].fillna(0)

# Create a field where any fatality or serious injury
# is classified as "high risk".
df["high_risk"] = ((si > 0) | (fat > 0)).astype(int)

In [4]:
#df["top_traffic_accident_offense"].unique()

In [5]:
# Drop columns. I am taking a guess by removing these
# columns which appear superflous. The leakage columns
# indicate data we are trying to predict, so we omit them.

ID_COLS = [
    "object_id", "incident_id", "offense_id",
    "incident_address", "reported_date",
    "last_occurrence_date", "POINT_X", "POINT_Y", "x", "y",
    "offense_code", "offense_code_extension",
]

LEAKAGE_COLS = [
    "SERIOUSLY_INJURED", "FATALITIES",
    "FATALITY_MODE_1", "FATALITY_MODE_2",
    "SERIOUSLY_INJURED_MODE_1", "SERIOUSLY_INJURED_MODE_2",
    "top_traffic_accident_offense",
    #"HARMFUL_EVENT_SEQ_1", "HARMFUL_EVENT_SEQ_2", "HARMFUL_EVENT_SEQ_3",
]

cols_to_drop = [c for c in ID_COLS + LEAKAGE_COLS]
df.drop(columns=cols_to_drop, inplace=True)

In [6]:
#print(df["first_occurrence_date"].unique())

In [7]:
# It is necessary to separate the time of the incident 
# into multiple categories.

df["first_occurrence_date"] = pd.to_datetime(
    df["first_occurrence_date"], errors="coerce"
)

df["hour"] = df["first_occurrence_date"].dt.hour
df["day_of_week"] = df["first_occurrence_date"].dt.dayofweek   # 0=Mon 6=Sun
df["month"] = df["first_occurrence_date"].dt.month

# We no longer need the raw datetime column
df.drop(columns=["first_occurrence_date"], inplace=True)

In [8]:
# Clean categorical columns

# selecting for obkect dtype gives us features with strings in them 
# These will be one hot encoded.
cat = df.select_dtypes(include="object").columns.tolist()

# Normalize strings
for col in cat:
    df[col] = (
        df[col]
        .astype(str)         # ensure string type
        .str.strip()         # remove leading/trailing whitespace
        .str.upper()         # normalize to uppercase
        .replace("NAN", np.nan)   # convert "nan" strings back to NaN
        .replace("", np.nan)      # blank strings -> NaN
    )

In [9]:
# Handle missing values

# Separate feature types (excluding the target)
feature_cols = [c for c in df.columns if c != "high_risk"]
cat_cols  = df[feature_cols].select_dtypes(include="object").columns.tolist()
num_cols  = df[feature_cols].select_dtypes(include=["number"]).columns.tolist()

# Fill categoricals with "UNKNOWN". By doing this, we can keep dataponits we would
# otherwise have to drop.
df[cat_cols] = df[cat_cols].fillna("UNKNOWN")

# Fill numeric columns with median. It's a safe choice.
for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)


In [15]:
# Separate features X and target y

X = df.drop(columns=["high_risk"])
y = df["high_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,      # keep class balance in both splits
)

# Re-identify column types from training data only
final_cat_cols = X_train.select_dtypes(include="object").columns.tolist()
final_num_cols = X_train.select_dtypes(include="number").columns.tolist()
print("Number columns")
print(final_num_cols)
print()
df["pedestrian_ind"].value_counts()

Number columns
['geo_x', 'geo_y', 'geo_lon', 'geo_lat', 'precinct_id', 'bicycle_ind', 'pedestrian_ind', 'hour', 'day_of_week', 'month']



pedestrian_ind
0.0    275919
1.0      6138
2.0       157
3.0        27
4.0         3
Name: count, dtype: int64

# Build ColumnTransformer preprocessor

preprocessor = ColumnTransformer(
    transformers=[
        # One-hot encode categorical columns; ignore unseen categories
        ("cat", OneHotEncoder(handle_unknown="ignore"), final_cat_cols),
        ("num", StandardScaler(), final_num_cols),
    ],
    remainder="drop",   # Discard any column not listed above.
)

In [13]:
def evaluate_model(name, pipeline, X_train, y_train, X_test, y_test, thr=0.5):
    """
    Parameters:
        string: name of classifier,
        Pipeline: classifier pipeline,
        X_train,
        y_train,
        X_test,
        y_test,
        int: threshold value (0..1)

    Returns:
        fitted pipeline,
        dictionary struct of important metrics
        
    Fit pipeline on training data, then print full evaluation on test data.
    ASSUME ALL MODELS HAVE predict_proba() FUNCTION
    
    """

    pipeline.fit(X_train, y_train)
    #y_pred = pipeline.predict(X_test)

    # generate the predictions using each models'
    # probability prediction for the samples. This 
    # way we can alter the threshold to improve recall
    # and we can use these models for a soft voter.
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= thr).astype(int)

    auc = roc_auc_score(y_test, y_prob)
    auc_str = f"{auc:.4f}"
    
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    cm   = confusion_matrix(y_test, y_pred)

    # Fancy printing done by Mr. Chat.
    print(f"\n{'─'*65}")
    print(f"  MODEL: {name}")
    print(f"{'─'*65}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}   (of predicted high-risk, how many truly are)")
    print(f"  Recall    : {rec:.4f}   (of actual high-risk, how many we caught)")
    print(f"  F1-Score  : {f1:.4f}   (harmonic mean of precision & recall)")
    print(f"  ROC-AUC   : {auc_str}")
    print()
    print("  Confusion matrix:")
    print(f"    {'':20s}  Pred=0   Pred=1")
    print(f"    Actual=0 (low risk)   {cm[0,0]:>6,}   {cm[0,1]:>6,}")
    print(f"    Actual=1 (high risk)  {cm[1,0]:>6,}   {cm[1,1]:>6,}")
    print()
    print("  Full classification report:")
    print(classification_report(y_test, y_pred, target_names=["Low risk", "High risk"]))

    return pipeline, {"name": name, "accuracy": acc, "precision": prec,
                      "recall": rec, "f1": f1, "auc": auc_str}

# Define Models

In [14]:
# Model 1, a dummy classifier that will always choose low risk.
# Acts as a control group.
dummy_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent", random_state=42)),
])

dummy_pipe_fitted, dummy_metrics = evaluate_model(
    "DummyClassifier (control)", dummy_pipe, X_train, y_train, X_test, y_test
)


─────────────────────────────────────────────────────────────────
  MODEL: DummyClassifier (control)
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.9767
  Precision : 0.0000   (of predicted high-risk, how many truly are)
  Recall    : 0.0000   (of actual high-risk, how many we caught)
  F1-Score  : 0.0000   (harmonic mean of precision & recall)
  ROC-AUC   : 0.5000

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   54,936        0
    Actual=1 (high risk)   1,308        0

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.98      1.00      0.99     54936
   High risk       0.00      0.00      0.00      1308

    accuracy                           0.98     56244
   macro avg       0.49      0.50      0.49     56244
weighted avg       0.95      0.98      0.97     56244



In [15]:
# Model 2, logistic regression

lr_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000, class_weight="balanced", C=0.01, random_state=42 # C was given to us through grid search. (thanks Mark)
    )),
])

lr_pipe_fitted, lr_metrics = evaluate_model(
    "Logistic Regression", lr_pipe, X_train, y_train, X_test, y_test
)


─────────────────────────────────────────────────────────────────
  MODEL: Logistic Regression
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.8286
  Precision : 0.0978   (of predicted high-risk, how many truly are)
  Recall    : 0.7752   (of actual high-risk, how many we caught)
  F1-Score  : 0.1738   (harmonic mean of precision & recall)
  ROC-AUC   : 0.8875

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   45,587    9,349
    Actual=1 (high risk)     294    1,014

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.99      0.83      0.90     54936
   High risk       0.10      0.78      0.17      1308

    accuracy                           0.83     56244
   macro avg       0.55      0.80      0.54     56244
weighted avg       0.97      0.83      0.89     56244



In [16]:
# Model 2, logistic regression with calibratedClassifier wrapper. 
#
#
# WIP IGNORE CELL

lr_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000, class_weight="balanced", C=0.01, random_state=42 # C was given to us through grid search.
    )),
])

calibrated_lr = CalibratedClassifierCV(
    estimator=lr_pipe,
    method="sigmoid",
    cv=5
)

lrc_pipe_fitted, lrc_metrics = evaluate_model(
    "Logistic Regression", calibrated_lr, X_train, y_train, X_test, y_test
)


─────────────────────────────────────────────────────────────────
  MODEL: Logistic Regression
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.9762
  Precision : 0.4336   (of predicted high-risk, how many truly are)
  Recall    : 0.0749   (of actual high-risk, how many we caught)
  F1-Score  : 0.1278   (harmonic mean of precision & recall)
  ROC-AUC   : 0.8872

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   54,808      128
    Actual=1 (high risk)   1,210       98

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.98      1.00      0.99     54936
   High risk       0.43      0.07      0.13      1308

    accuracy                           0.98     56244
   macro avg       0.71      0.54      0.56     56244
weighted avg       0.97      0.98      0.97     56244



In [17]:
# Model 3, Random Forest 
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=5,
    )),
])

rf_pipe_fitted, rf_metrics = evaluate_model(
    "Random Forest", rf_pipe, X_train, y_train, X_test, y_test
)



─────────────────────────────────────────────────────────────────
  MODEL: Random Forest
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.9614
  Precision : 0.2624   (of predicted high-risk, how many truly are)
  Recall    : 0.3639   (of actual high-risk, how many we caught)
  F1-Score  : 0.3049   (harmonic mean of precision & recall)
  ROC-AUC   : 0.8880

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   53,598    1,338
    Actual=1 (high risk)     832      476

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.98      0.98      0.98     54936
   High risk       0.26      0.36      0.30      1308

    accuracy                           0.96     56244
   macro avg       0.62      0.67      0.64     56244
weighted avg       0.97      0.96      0.96     56244



In [18]:
# Extra Tree Classifier
et_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", ExtraTreesClassifier(
        n_estimators=500,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
        max_depth=20,
        min_samples_leaf=5,
    )),
])
et_pipe.fit(X_train, y_train)
et_probs = et_pipe.predict_proba(X_test)[:, 1]

et_pipe_fitted, et_metrics = evaluate_model(
    "Random Forest", et_pipe, X_train, y_train, X_test, y_test
)


─────────────────────────────────────────────────────────────────
  MODEL: Random Forest
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.8463
  Precision : 0.1024   (of predicted high-risk, how many truly are)
  Recall    : 0.7225   (of actual high-risk, how many we caught)
  F1-Score  : 0.1794   (harmonic mean of precision & recall)
  ROC-AUC   : 0.8734

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   46,652    8,284
    Actual=1 (high risk)     363      945

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.99      0.85      0.92     54936
   High risk       0.10      0.72      0.18      1308

    accuracy                           0.85     56244
   macro avg       0.55      0.79      0.55     56244
weighted avg       0.97      0.85      0.90     56244



In [19]:
# Build a fresh pipeline with all three estimators inside a VotingClassifier.
# Each estimator gets its own preprocessing step baked in.
from sklearn.pipeline import Pipeline as Pipe

def lr_sub():
    return Pipe([("pre", preprocessor),
                 ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", C=0.01, random_state=42, n_jobs=-1))])

def rf_sub():
    return Pipe([("pre", preprocessor),
                 ("clf", RandomForestClassifier(n_estimators=500, class_weight="balanced_subsample",
                                                random_state=42, n_jobs=-1, max_depth=20, min_samples_leaf=5))])

def et_sub():
    return Pipe([("pre", preprocessor),
                 ("clf", ExtraTreesClassifier(n_estimators=500, class_weight="balanced_subsample",
                                             random_state=42, n_jobs=-1, max_depth=20, min_samples_leaf=5))])

voting_pipe = VotingClassifier(
    estimators=[("lr", lr_sub()), ("rf", rf_sub()), ("et", et_sub())],
    voting="soft",   # average probabilities, not just votes
    n_jobs=-1,
)

vote_pipe_fitted, vote_metrics = evaluate_model(
    "Voting", voting_pipe, X_train, y_train, X_test, y_test, thr=0.4)


─────────────────────────────────────────────────────────────────
  MODEL: Voting
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.7323
  Precision : 0.0705   (of predicted high-risk, how many truly are)
  Recall    : 0.8631   (of actual high-risk, how many we caught)
  F1-Score  : 0.1304   (harmonic mean of precision & recall)
  ROC-AUC   : 0.8853

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   40,061   14,875
    Actual=1 (high risk)     179    1,129

  Full classification report:
              precision    recall  f1-score   support

    Low risk       1.00      0.73      0.84     54936
   High risk       0.07      0.86      0.13      1308

    accuracy                           0.73     56244
   macro avg       0.53      0.80      0.49     56244
weighted avg       0.97      0.73      0.83     56244



In [ ]:
#Adaboost as well